# Earthquake Magnitude Analysis — EDA

**Name:**  
**Dataset:** USGS Earthquake Events (2023–2024, magnitude ≥ 1.5)  
**Dependent Variable:** `mag` (earthquake magnitude, quantitative)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os

sns.set_theme(style='whitegrid')

df = pd.read_csv('../data/earthquakes.csv')
print(df.shape)
df.head()

## 1. Basic Summary Statistics

In [ ]:
df.describe()

In [ ]:
df.info()

In [ ]:
print('Missing values per column:')
df.isnull().sum().sort_values(ascending=False)

## 2. Distribution of Dependent Variable (mag)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

axes[0].hist(df['mag'].dropna(), bins=40, edgecolor='black', color='steelblue')
axes[0].set_title('Distribution of Earthquake Magnitude')
axes[0].set_xlabel('Magnitude')
axes[0].set_ylabel('Count')

axes[1].boxplot(df['mag'].dropna())
axes[1].set_title('Boxplot of Earthquake Magnitude')
axes[1].set_ylabel('Magnitude')

plt.tight_layout()
os.makedirs('../results', exist_ok=True)
plt.savefig('../results/mag_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

## 3. Categorical Variables vs. Magnitude

In [ ]:
cat_cols = ['magType', 'type', 'status']

for col in cat_cols:
    top_cats = df[col].value_counts().nlargest(10).index
    subset = df[df[col].isin(top_cats)]

    plt.figure(figsize=(10, 4))
    sns.boxplot(data=subset, x=col, y='mag', order=top_cats)
    plt.title(f'Magnitude by {col}')
    plt.xticks(rotation=45)
    plt.tight_layout()
    plt.savefig(f'../results/{col}_vs_mag.png', dpi=150, bbox_inches='tight')
    plt.show()

## 4. Quantitative Variables vs. Magnitude (Scatter Plots)

In [ ]:
quant_cols = ['depth', 'nst', 'gap', 'dmin', 'rms']

fig, axes = plt.subplots(2, 3, figsize=(15, 8))
axes = axes.flatten()

for i, col in enumerate(quant_cols):
    axes[i].scatter(df[col], df['mag'], alpha=0.1, s=5, color='steelblue')
    axes[i].set_xlabel(col)
    axes[i].set_ylabel('mag')
    axes[i].set_title(f'{col} vs. Magnitude')

axes[-1].set_visible(False)
plt.tight_layout()
plt.savefig('../results/quant_vs_mag.png', dpi=150, bbox_inches='tight')
plt.show()

## 5. Correlation Heatmap

In [ ]:
num_df = df[['mag', 'depth', 'latitude', 'longitude', 'nst', 'gap', 'dmin', 'rms', 'month', 'year']].dropna()

plt.figure(figsize=(10, 8))
sns.heatmap(num_df.corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1)
plt.title('Correlation Heatmap of Numeric Variables')
plt.tight_layout()
plt.savefig('../results/correlation_heatmap.png', dpi=150, bbox_inches='tight')
plt.show()

## 6. Geographic Distribution

In [ ]:
plt.figure(figsize=(14, 6))
scatter = plt.scatter(
    df['longitude'], df['latitude'],
    c=df['mag'], cmap='hot_r', alpha=0.3, s=5
)
plt.colorbar(scatter, label='Magnitude')
plt.title('Global Earthquake Map (2023–2024)')
plt.xlabel('Longitude')
plt.ylabel('Latitude')
plt.tight_layout()
plt.savefig('../results/earthquake_map.png', dpi=150, bbox_inches='tight')
plt.show()

## 7. Earthquakes per Month

In [ ]:
monthly = df.groupby(['year', 'month']).size().reset_index(name='count')
monthly['year_month'] = monthly['year'].astype(str) + '-' + monthly['month'].astype(str).str.zfill(2)

plt.figure(figsize=(14, 4))
plt.bar(monthly['year_month'], monthly['count'], color='steelblue')
plt.xticks(rotation=90)
plt.title('Number of Earthquakes per Month (2023–2024)')
plt.xlabel('Year-Month')
plt.ylabel('Count')
plt.tight_layout()
plt.savefig('../results/earthquakes_per_month.png', dpi=150, bbox_inches='tight')
plt.show()